In [1]:
import nibabel as nib
import os
import matplotlib.pyplot as plt
import numpy as np
import pydicom
import pandas as pd
from ipywidgets import Dropdown, IntSlider, VBox, Output, Label, interactive
from IPython.display import display, clear_output
from datetime import datetime

from utils.dcm2niix import *
from utils.selection import select_modalities

# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'
APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_selection.log')


# THIRD TRY preselection by hakim martin and julian

In [2]:
dwi_t1_df_3=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_merged.csv"))
dwi_t1_df_3.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_3.head()
len(dwi_t1_df_3)

dwi_t1_df_3.groupby('Patient_ID').nunique().head()

,Study_ID,Sequence_ID,File_Name,Full_Path,Modality,Date,Sequence_ID_split
Patient_ID,,,,,,,
76813,2,2,50,79,2,2,2
442968,2,2,38,62,2,2,2
481858,1,2,40,64,2,1,2
516341,2,2,160,204,2,2,2
653853,1,2,50,74,2,1,2


In [3]:
#dwi
df_dwi = dwi_t1_df_3[dwi_t1_df_3['Modality']=='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_3[dwi_t1_df_3['Modality']!='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['anatomic'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

/nasdata/Berner_Daten/Other_Side_Bilddaten/0002923041/20170703_4379063/Diffusion_tra_3mm_TRACEW__1.3.12.2.1107.5.2.18.141120.2017070310463983788943340.0.0.0
/home/bule/berner_daten_analyse/data/nifti_third_try/0002923041/20170703_4379063/Diffusion_tra_3mm_TRACEW__1.3.12.2.1107.5.2.18.141120.2017070310463983788943340.0.0.0
/nasdata/Berner_Daten/Other_Side_Bilddaten/0011597844/20130528_3155306/t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.2013052816375883690366642.0.0.0
/home/bule/berner_daten_analyse/data/nifti_third_try/0011597844/20130528_3155306/t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.2013052816375883690366642.0.0.0


,Path_Nifti,Modality,join_crit
0,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,/0002923041/20170703_4379063/Diffusion_tra_3mm...
1,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,/0002961156/20120727_2957276/ep2d_diff_3scan_t...
2,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,/0007815360/20140614_3430160/ep2d_diff_3scan_t...
3,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,/0011420316/20130204_3078550/ep2d_diff_M128_b0...
4,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,/0082395993/20170802_4406866/ep2d_diff_M128_b0...


In [ ]:
# check for duplicated  select all created niftis -> some didnt work
data=[]
for path in all_niftis:
    patient_id = int(path.split("/")[6])
    study_id = path.split("/")[7]
    sequence_id = path.split("/")[8]
    full_path = path
    data.append({'Patient_ID':patient_id, 'Study_ID':study_id,'Sequence_ID':sequence_id ,'Full_Path_Nifti':full_path})
    
nii_df=pd.DataFrame(data)
nii_df['join_crit']=[ os.path.join('/',*path.split('/')[6:-1]) for path in nii_df['Full_Path_Nifti']]
nii_df=nii_df.merge(df_modality, on='join_crit', how='left')

# len(nii_df[['Sequence_ID','Study_ID']].unique()) 
duplicates = nii_df[nii_df.duplicated(subset=['Study_ID', 'Sequence_ID'], keep=False)]
print(duplicates.head(1))

nii_df.drop_duplicates(subset=['Sequence_ID'], keep='first', inplace=True)
nii_df.info()

     Patient_ID          Study_ID  \
457    13886029  20170211_4244906   

                                           Sequence_ID  \
457  ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...   

                                       Full_Path_Nifti  \
457  /home/bule/berner_daten_analyse/data/nifti_thi...   

                                             join_crit  \
457  /0013886029/20170211_4244906/ep2d_diff_3scan_t...   

                                            Path_Nifti Modality  
457  /home/bule/berner_daten_analyse/data/nifti_thi...      dwi  
<class 'pandas.core.frame.DataFrame'>
Index: 855 entries, 0 to 859
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Patient_ID       855 non-null    int64 
 1   Study_ID         855 non-null    object
 2   Sequence_ID      855 non-null    object
 3   Full_Path_Nifti  855 non-null    object
 4   join_crit        855 non-null    object
 5   Path_Nifti       855 no

In [7]:
nii_df = nii_df.sort_values(by='Patient_ID')
nii_df['Date'] = nii_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
nii_df

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality,Date
451,76813,20170929_4371316,t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000076813/20170929_4371316/t1_se_tra__1.3.12...,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-09-29
452,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000076813/20170614_4361718/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2017-06-14
25,442968,20151110_3835468,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000442968/20151110_3835468/t1_se_r_tra_KM__1...,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2015-11-10
24,442968,20151109_3835318,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000442968/20151109_3835318/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2015-11-09
542,481858,20121025_3013771,resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000481858/20121025_3013771/resolve_3scan_tra...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2012-10-25
...,...,...,...,...,...,...,...,...
772,82364672,20170113_4216439,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.201701...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082364672/20170113_4216439/t1_fl2d_tra__1.3....,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-01-13
656,82395993,20170802_4406866,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.201708...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082395993/20170802_4406866/t1_fl2d_tra__1.3....,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-08-02
657,82395993,20170802_4406866,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082395993/20170802_4406866/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2017-08-02
48,82789223,20161003_4120929,resolve_3scan_trace_tra_p3_160_TRACEW__1.3.12....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082789223/20161003_4120929/resolve_3scan_tra...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2016-10-03


In [9]:
## save to df for skullstripping 2 sequences per patient
nii_df = nii_df.sort_values(by='Patient_ID')
nii_df['Date'] = nii_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

only_two_df = nii_df.groupby('Patient_ID', group_keys=False).apply(select_modalities)
only_two_df['Shape']=only_two_df['Full_Path_Nifti'].apply(lambda x: nib.load(x).shape)

only_two_df.reset_index(drop=True, inplace=True)
only_two_df['Modality'].value_counts()

/tmp/ipykernel_8534/648453607.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  only_two_df = nii_df.groupby('Patient_ID', group_keys=False).apply(select_modalities)


Modality
dwi         425
anatomic    425
Name: count, dtype: int64

In [10]:
only_two_df

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality,Date,Shape
0,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000076813/20170614_4361718/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2017-06-14,"(128, 128, 25, 2)"
1,76813,20170929_4371316,t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000076813/20170929_4371316/t1_se_tra__1.3.12...,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-09-29,"(600, 640, 29)"
2,442968,20151109_3835318,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000442968/20151109_3835318/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2015-11-09,"(192, 192, 19, 2)"
3,442968,20151110_3835468,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000442968/20151110_3835468/t1_se_r_tra_KM__1...,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2015-11-10,"(224, 256, 24)"
4,481858,20121025_3013771,resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0000481858/20121025_3013771/resolve_3scan_tra...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2012-10-25,"(176, 176, 20, 2)"
...,...,...,...,...,...,...,...,...,...
845,82364672,20170113_4216439,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.201701...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082364672/20170113_4216439/t1_fl2d_tra__1.3....,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-01-13,"(336, 384, 24)"
846,82395993,20170802_4406866,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082395993/20170802_4406866/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2017-08-02,"(128, 128, 25, 2)"
847,82395993,20170802_4406866,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.201708...,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082395993/20170802_4406866/t1_fl2d_tra__1.3....,/home/bule/berner_daten_analyse/data/nifti_thi...,anatomic,2017-08-02,"(324, 384, 24)"
848,82789223,20161003_4120929,resolve_3scan_trace_tra_p3_160_TRACEW__1.3.12....,/home/bule/berner_daten_analyse/data/nifti_thi...,/0082789223/20161003_4120929/resolve_3scan_tra...,/home/bule/berner_daten_analyse/data/nifti_thi...,dwi,2016-10-03,"(160, 160, 30, 2)"


In [11]:

only_two_df.to_csv(os.path.join(ANALYSIS_PATH, 'only2_third_try_df.csv'), index=False)

# Test Pascal

In [2]:
dwi_t1_df_pascal_test=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_pascal_test.csv"))
dwi_t1_df_pascal_test.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_pascal_test.head()
dwi_t1_df_pascal_test.groupby('Patient_ID').nunique().head(2)

,Study_ID,Sequence_ID,File_Name,Full_Path,Modality,Date,Sequence_ID_split
Patient_ID,,,,,,,
76813,2,2,50,79,2,2,2
442968,1,2,38,62,2,1,2


In [3]:
#dwi
df_dwi = dwi_t1_df_pascal_test[dwi_t1_df_pascal_test['Modality']=='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_pascal_test[dwi_t1_df_pascal_test['Modality']!='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['anatomic'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality


/nasdata/Berner_Daten/Other_Side_Bilddaten/0003327876/20141218_3572501/resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12.2.1107.5.2.18.41437.2014121815344685225286636.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0003327876/20141218_3572501/resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12.2.1107.5.2.18.41437.2014121815344685225286636.0.0.0
/nasdata/Berner_Daten/Other_Side_Bilddaten/0001428012/20160927_4117379/t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.2016092715275828226260551.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001428012/20160927_4117379/t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.2016092715275828226260551.0.0.0


In [6]:
# check for duplicated  select all created niftis -> some didnt work
data=[]
for path in all_niftis:
    patient_id = int(path.split("/")[6])
    study_id = path.split("/")[7]
    sequence_id = path.split("/")[8]
    full_path = path
    data.append({'Patient_ID':patient_id, 'Study_ID':study_id,'Sequence_ID':sequence_id ,'Full_Path_Nifti':full_path})
    
nii_df=pd.DataFrame(data)
nii_df['join_crit']=[ os.path.join('/',*path.split('/')[6:-1]) for path in nii_df['Full_Path_Nifti']]
nii_df=nii_df.merge(df_modality, on='join_crit', how='left')

# len(nii_df[['Sequence_ID','Study_ID']].unique()) 
duplicates = nii_df[nii_df.duplicated(subset=['Study_ID', 'Sequence_ID'], keep=False)]
print(duplicates.head(1))

nii_df.drop_duplicates(subset=['Sequence_ID'], keep='first', inplace=True)
nii_df.info()

Empty DataFrame
Columns: [Patient_ID, Study_ID, Sequence_ID, Full_Path_Nifti, join_crit, Path_Nifti, Modality]
Index: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Patient_ID       249 non-null    int64 
 1   Study_ID         249 non-null    object
 2   Sequence_ID      249 non-null    object
 3   Full_Path_Nifti  249 non-null    object
 4   join_crit        249 non-null    object
 5   Path_Nifti       249 non-null    object
 6   Modality         249 non-null    object
dtypes: int64(1), object(6)
memory usage: 13.7+ KB


In [5]:
## save to df for skullstripping 2 sequences per patient
nii_df = nii_df.sort_values(by='Patient_ID')
nii_df['Date'] = nii_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

only_two_df = nii_df.groupby('Patient_ID', group_keys=False).apply(select_modalities)
only_two_df.reset_index(drop=True, inplace=True)
only_two_df.to_csv(os.path.join(ANALYSIS_PATH, 'only2_pascal_selection_df.csv'), index=False)

/tmp/ipykernel_793900/602695824.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  only_two_df = nii_df.groupby('Patient_ID', group_keys=False).apply(select_modalities)


# Second try

In [9]:
dwi_t1_df_2=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_2.csv"))
dwi_t1_df_2.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_2.head()
len(dwi_t1_df_2)

dwi_t1_df_2.groupby('Patient_ID').nunique().head()

,Study_ID,Sequence_ID,File_Name,Full_Path,Date,Sequence_ID_split,Modality
Patient_ID,,,,,,,
76813,3,4,50,167,3,3,2
442968,2,2,38,76,2,1,1
516341,2,5,160,302,2,3,2
607681,2,3,176,230,2,3,1
653853,4,6,261,455,4,4,2


In [4]:
#dwi
df_dwi = dwi_t1_df_2[dwi_t1_df_2['Modality']=='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_2[dwi_t1_df_2['Modality']!='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['anatomic'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

/nasdata/Berner_Daten/Other_Side_Bilddaten/0005146992/20151022_3820341/resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12.2.1107.5.2.18.41437.2015102213042729181883340.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0005146992/20151022_3820341/resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12.2.1107.5.2.18.41437.2015102213042729181883340.0.0.0
/nasdata/Berner_Daten/Other_Side_Bilddaten/0013762940/20161209_4186263/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.2016120919411033123326420.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0013762940/20161209_4186263/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.2016120919411033123326420.0.0.0


,Path_Nifti,Modality,join_crit
0,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi,/0005146992/20151022_3820341/resolve_3scan_tra...
1,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi,/0002840065/20141219_3502193/ep2d_diff_M128_b0...
2,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi,/0001775707/20150301_3626826/ep2d_diff_3scan_t...
3,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi,/0003354261/20170524_4343300/ep2d_diff_3scan_t...
4,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi,/0011597844/20130528_3155306/resolve_3scan_tra...


In [ ]:
#dwi
df_dwi = dwi_t1_df_2[dwi_t1_df_2['Modality']=='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_2[dwi_t1_df_2['Modality']!='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['anatomic'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

/nasdata/Berner_Daten/Other_Side_Bilddaten/0001829939/20150718_3741531/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2015071807190941239701338.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001829939/20150718_3741531/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2015071807190941239701338.0.0.0
/nasdata/Berner_Daten/Other_Side_Bilddaten/0013361457/20160812_3986923/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.2016081211224635972737391.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0013361457/20160812_3986923/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.2016081211224635972737391.0.0.0


In [5]:
data=[]
for path in all_niftis:
    patient_id = int(path.split("/")[6])
    study_id = path.split("/")[7]
    sequence_id = path.split("/")[8]
    full_path = path
    data.append({'Patient_ID':patient_id, 'Study_ID':study_id,'Sequence_ID':sequence_id ,'Full_Path_Nifti':full_path})
    
nii_df=pd.DataFrame(data)
nii_df['join_crit']=[ os.path.join('/',*path.split('/')[6:-1]) for path in nii_df['Full_Path_Nifti']]

nii_df=nii_df.merge(df_modality, on='join_crit', how='left')

# len(nii_df[['Sequence_ID','Study_ID']].unique()) 
duplicates = nii_df[nii_df.duplicated(subset=['Study_ID', 'Sequence_ID'], keep=False)]
duplicates

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality
0,1029355,20150421_3667803,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
1,1029355,20150421_3667803,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
2,1029355,20150421_3667803,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/t1_se_r_tra_KM__1...,NaN,NaN
3,1029355,20150421_3667803,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/t1_se_r_tra_KM__1...,NaN,NaN
4,3082016,20150215_3615242,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0003082016/20150215_3615242/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
...,...,...,...,...,...,...,...
484,1489860,20120330_2884717,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001489860/20120330_2884717/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
485,1424025,20170712_4389337,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001424025/20170712_4389337/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
486,1424025,20170712_4389337,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001424025/20170712_4389337/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
487,1424025,20170712_4389337,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.201707...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001424025/20170712_4389337/t1_fl2d_tra__1.3....,/home/bule/berner_daten_analyse/data/nifti_pas...,t1


- somehow there  are some duplicatet t1s 

In [6]:
nii_df.drop_duplicates(subset=['Sequence_ID'], keep='first', inplace=True)
nii_df

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality
0,1029355,20150421_3667803,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
2,1029355,20150421_3667803,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/t1_se_r_tra_KM__1...,NaN,NaN
4,3082016,20150215_3615242,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0003082016/20150215_3615242/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
6,3082016,20150215_3615242,t1_se_tra__1.3.12.2.1107.5.2.30.27272.20150215...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0003082016/20150215_3615242/t1_se_tra__1.3.12...,/home/bule/berner_daten_analyse/data/nifti_pas...,t1
8,2782782,20160119_3893609,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0002782782/20160119_3893609/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
...,...,...,...,...,...,...,...
479,2593122,20160922_4112749,t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.20160...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0002593122/20160922_4112749/t1_tse_r_tra__1.3...,/home/bule/berner_daten_analyse/data/nifti_pas...,t1
481,1489860,20120330_2884717,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001489860/20120330_2884717/t1_se_r_tra_KM__1...,NaN,NaN
483,1489860,20120330_2884717,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001489860/20120330_2884717/ep2d_diff_3scan_t...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi
485,1424025,20170712_4389337,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001424025/20170712_4389337/ep2d_diff_M128_b0...,/home/bule/berner_daten_analyse/data/nifti_pas...,dwi


- OLEA analysis seems to make problems here -> removed them 

In [ ]:
file_path='/home/bule/berner_daten_analyse/data/nifti_second_try/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0/ep2d_diff_M128_b0_1000_DIN_1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0_ep2d_diff_M128_b0_1000_DIN_20170614164701_3.nii.gz'
img = nib.load(file_path)
data_dwi = img.get_fdata()
def view_slice_dwi(slice_idx):
    clear_output(wait=True)  # Clear previous output to avoid duplicate plots
    plt.figure(figsize=(3, 3))
    plt.imshow(data[:, :, slice_idx,1].squeeze(), cmap="gray", origin="lower")
    plt.title(f"Slice {slice_idx}")
    plt.axis("off")
    plt.show()
slider_dwi = IntSlider(min=0, max=data_dwi.shape[2] - 1, step=1, value=0)
interactive_plot_dwi = interactive(view_slice_dwi, slice_idx=slider_dwi)
display(interactive_plot_dwi)

In [ ]:
file_path='/home/bule/berner_daten_analyse/data/nifti_second_try/0000076813/20170929_4371316/t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092912262239073557517.0.0.0/t1_se_tra_1.3.12.2.1107.5.2.41.169589.2017092912262239073557517.0.0.0_t1_se_tra_20170929120431_19.nii.gz'
img = nib.load(file_path)
data = img.get_fdata()
def view_slice(slice_idx):
    clear_output(wait=True)  # Clear previous output to avoid duplicate plots
    plt.figure(figsize=(3, 3))
    plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
    plt.title(f"Slice {slice_idx}")
    plt.axis("off")
    plt.show()
    
slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
interactive_plot = interactive(view_slice, slice_idx=slider)
display(interactive_plot)

In [ ]:
df = nii_df.sort_values(by='Patient_ID')

df['Date'] = df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

pd.set_option('display.max_colwidth', None)

patient_dropdown = Dropdown(options=df['Patient_ID'].unique(), description='Patient:')
sequence_dropdown = Dropdown(description='Sequence:')
sequences_output = Output()
slider_output = Output()  
plot_output = Output()    

def update_sequences(*args):
    patient_id = patient_dropdown.value
    patient_data = df[df['Patient_ID'] == patient_id]
    sequence_dropdown.options = patient_data['Sequence_ID'].values

    with sequences_output:
        clear_output(wait=True)
        display(patient_data[['Date', 'Sequence_ID', 'Modality']].sort_values(by='Date'))

def display_modality(*args):
    patient_id = patient_dropdown.value
    sequence_id = sequence_dropdown.value
    if sequence_id is None:
        return
    row = df[(df['Patient_ID'] == patient_id) & (df['Sequence_ID'] == sequence_id)].iloc[0]
    file_path = row['Full_Path_Nifti']
    modality = row['Modality']

    img = nib.load(file_path)
    data = img.get_fdata()
    if data.ndim == 4:
        data=data[:,:,:,1]
        
    def view_slice(slice_idx):
        with plot_output:
            clear_output(wait=True)
            plt.figure(figsize=(5, 5))
            plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
            plt.title(f"Slice {slice_idx}")
            plt.axis("off")
            plt.show()
    slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
    slider.observe(lambda change: view_slice(slider.value), names='value')

    with slider_output:
        clear_output(wait=True)
        display(slider)
    view_slice(0)

patient_dropdown.observe(update_sequences, names='value')
sequence_dropdown.observe(display_modality, names='value')

update_sequences()

display(VBox([
    patient_dropdown,
    sequences_output,
    sequence_dropdown,
    slider_output,
    plot_output
]))




In [ ]:
## save to df for skullstripping 2 sequences per patient
def select_modalities(patient_data):
    dwi = patient_data[patient_data['Modality'] == 'dwi'].sort_values(by='Date').head(1)
    t1 = patient_data[patient_data['Modality'] == 't1'].sort_values(by='Date').head(1)
    if not dwi.empty and not t1.empty:
        return pd.concat([dwi, t1])
    return pd.DataFrame()  

only_two_df = df.groupby('Patient_ID', group_keys=False).apply(select_modalities)
only_two_df.reset_index(drop=True, inplace=True)
print(only_two_df.groupby('Patient_ID').nunique())

only_two_df.to_csv(os.path.join(ANALYSIS_PATH, 'only2_df.csv'), index=False)

### Compare to Dicom


- seems to fit

In [ ]:
# dicom_data = pydicom.dcmread('/nasdata/Berner_Daten/Other_Side_Bilddaten/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0/15.dcm'
# , )  # Load without pixel data to check header only
# dicom_data.pixel_array.shape

# plt.imshow(dicom_data.pixel_array, cmap="gray", origin="lower")

In [ ]:
# dicom_data = pydicom.dcmread('/nasdata/Berner_Daten/Other_Side_Bilddaten/0013140710/20160101_3878363/B1000_OLEA_ANALYSIS__1.3.6.1.4.1.32839.99.2.228328515211608092063513064401162864980/2.dcm'
# , )  # Load without pixel data to check header only
# dicom_data.pixel_array.shape

# plt.imshow(dicom_data.pixel_array, cmap="gray", origin="lower")